# Session 2 · Part 5 — Visualize inferred protein landscapes

**Goal:** inspect prediction distributions and spatial patterns before calculating accuracy metrics.
A plausible-looking map is not proof of correctness; it is a diagnostic for range compression, isolated
artifacts, tissue-edge effects, and expected regional structure.


In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
for candidate in (current, *current.parents):
    if (candidate / "src" / "dgat_tutorial").is_dir():
        tutorial_root = candidate
        break
else:
    raise FileNotFoundError("Start Jupyter inside the hands-on_tutorial directory.")

sys.path.insert(0, str(tutorial_root / "src"))

from dgat_tutorial.checkpoints import tutorial_paths, write_checkpoint

paths = tutorial_paths(tutorial_root)
print(f"Tutorial root: {paths.root}")


## 1. Align predictions to spatial coordinates


In [ ]:
import matplotlib.pyplot as plt

from dgat_tutorial.checkpoints import preferred_prediction_path
from dgat_tutorial.data import load_tutorial_data
from dgat_tutorial.dgat import load_prediction_table
from dgat_tutorial.plotting import plot_spatial_feature

dataset = load_tutorial_data(paths.raw_data)
prediction_path = preferred_prediction_path(paths)
predicted = load_prediction_table(str(prediction_path))
common_spots = dataset.spots.index.intersection(predicted.index)
if common_spots.empty:
    raise ValueError("Spatial data and predictions have no shared spot IDs; check that the assets match.")
spots = dataset.spots.loc[common_spots]
predicted = predicted.loc[common_spots]
print(f"Aligned {len(common_spots)} spots and {predicted.shape[1]} predicted proteins")


### Figure 8 — Prediction distributions


In [ ]:
proteins_to_plot = list(predicted.var(axis=0).nlargest(min(4, predicted.shape[1])).index)
fig, axes = plt.subplots(1, len(proteins_to_plot), figsize=(3.2 * len(proteins_to_plot), 3.2))
axes = [axes] if len(proteins_to_plot) == 1 else axes
for ax, protein in zip(axes, proteins_to_plot):
    ax.hist(predicted[protein], bins=30, color="#4c78a8")
    ax.set_title(protein); ax.set_xlabel("predicted abundance"); ax.set_ylabel("spots")
distribution_path = paths.figures / "session02_prediction_distributions.png"
fig.tight_layout(); fig.savefig(distribution_path, dpi=160, bbox_inches="tight"); plt.show()


### Figure 9 — Predicted spatial protein maps


In [ ]:
n_cols = 2
n_rows = (len(proteins_to_plot) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(9, 4 * n_rows), squeeze=False)
for ax, protein in zip(axes.ravel(), proteins_to_plot):
    plot_spatial_feature(spots, predicted[protein], f"Predicted {protein}", ax=ax)
for ax in axes.ravel()[len(proteins_to_plot):]: ax.axis("off")
spatial_path = paths.figures / "session02_predicted_protein_maps.png"
fig.tight_layout(); fig.savefig(spatial_path, dpi=160, bbox_inches="tight"); plt.show()


In [ ]:
manifest = write_checkpoint(
    "2.5", [prediction_path, distribution_path, spatial_path],
    summary={"spots": len(common_spots), "proteins_plotted": proteins_to_plot}, start=paths.root,
)
print(f"Checkpoint written: {manifest}")


## Check

Identify one map worth evaluating and one possible artifact. Then continue to Session 3, where observed
proteins are used for pointwise correlation, spatial coherence, and side-by-side landscape comparison.
